# Проверка смешанной симуляции NovaSeq для человека

Проверяется ветвь `results/PRJEB30386/simulated/insilicoseq_150bp_novaseq_post_annotation_filtered/`, объединяющая библиотеки IgM/IGH, IgG/IGH, IgK/IGK и IgL/IGL в один вычислительный образец для TRUST4. Контроль включает: (1) число пар, длину PE150, распределение ридов и сохранение числа молекул PCR1→фрагментация; (2) соответствие бюджета IGH/IGK/IGL заданной глубине; (3) выравнивание детерминированной подвыборки на точные фрагменты и, при необходимости, исходные шаблоны V–J.


In [ ]:
import csv, gzip, json, math, os, re, shutil, subprocess, sys, sysconfig, time
from collections import Counter, defaultdict
from pathlib import Path

DATASET = "PRJEB30386"
BRANCH = "insilicoseq_150bp_novaseq_post_annotation_filtered"
SAMPLE = "PRJEB30386_all_chains"

# Для быстрой проверки достаточно 250 тыс. равномерно выбранных пар ридов.
# Значение None включает выравнивание всех 4,55 млн пар.
VALIDATION_PAIRS = 250_000

# Выравнивание на шаблоны информативно, но неоднозначнее выравнивания на фрагменты,
# поскольку многие BCR-шаблоны содержат гомологичные последовательности.
RUN_TEMPLATE_ALIGNMENT = True

NPROC = 8
FORCE = False

def resolve_root():
    candidates = []
    if os.environ.get("BCR_VOLUME"):
        candidates.append(Path(os.environ["BCR_VOLUME"]))
    candidates += [
        Path("/data/user/epishkin"),
        Path("/Users/epishkin/workspace/bcr-assembler"),
    ]
    start = Path.cwd().resolve()
    candidates += [start, *start.parents]

    seen = set()
    for root in candidates:
        if str(root) in seen:
            continue
        seen.add(str(root))
        if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
            return root
    raise FileNotFoundError(
        f"Cannot locate results/{DATASET}/simulated/{BRANCH}. "
        "Rename the completed branch first or set BCR_VOLUME."
    )

ROOT = resolve_root()
SIM_DIR = ROOT / "results" / DATASET / "simulated" / BRANCH

TRUTH_DIR = SIM_DIR / "00_primary_truth"
PCR1_DIR = SIM_DIR / "01_pcr1"
FRAG_DIR = SIM_DIR / "02_fragmentation"
PCR2_DIR = SIM_DIR / "03_pcr2"
ALLOC_DIR = SIM_DIR / "04_read_allocation"
FASTQ_DIR = SIM_DIR / "06_fastq_pe150"
SIM_QC_DIR = SIM_DIR / "qc"

VALIDATION_DIR = SIM_DIR / "validation"
SUBSET_DIR = VALIDATION_DIR / "subset_fastq"
FRAG_ALIGN_DIR = VALIDATION_DIR / "fragment_alignment"
TEMPLATE_ALIGN_DIR = VALIDATION_DIR / "template_alignment"

for d in (VALIDATION_DIR, SUBSET_DIR, FRAG_ALIGN_DIR, TEMPLATE_ALIGN_DIR):
    d.mkdir(parents=True, exist_ok=True)

R1 = FASTQ_DIR / f"{SAMPLE}_R1.fastq.gz"
R2 = FASTQ_DIR / f"{SAMPLE}_R2.fastq.gz"
FRAGMENT_FASTA = ALLOC_DIR / f"{SAMPLE}_selected_fragments.fasta"
ALLOCATION_TSV = ALLOC_DIR / f"{SAMPLE}_allocation.tsv"
PRIMARY_TEMPLATES = TRUTH_DIR / f"{SAMPLE}_templates.fasta"
PCR1_TSV = PCR1_DIR / f"{SAMPLE}_pcr1_pool.tsv"
FRAGMENT_TSV = FRAG_DIR / f"{SAMPLE}_fragments.tsv"
FINAL_QC = SIM_QC_DIR / "final_qc.tsv"

for p in (R1, R2, FRAGMENT_FASTA, ALLOCATION_TSV, PRIMARY_TEMPLATES, PCR1_TSV, FRAGMENT_TSV, FINAL_QC):
    if not p.exists():
        raise FileNotFoundError(p)

for tool in ("bowtie2", "bowtie2-build", "samtools"):
    if not shutil.which(tool):
        raise RuntimeError(f"{tool} not found in PATH")

print("ROOT:", ROOT)
print("SIM_DIR:", SIM_DIR)
print("Validation sample:", SAMPLE)


## 1. Внутренние инварианты симуляции

Ключевые условия симуляции проверяются независимо.


In [ ]:
def read_tsv(path):
    with open(path, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))

def count_fastq(path):
    n = 0
    lengths = set()
    with gzip.open(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                break
            seq = h.readline().rstrip("\r\n")
            plus = h.readline()
            qual = h.readline().rstrip("\r\n")
            if not qual or not head.startswith("@") or not plus.startswith("+"):
                raise RuntimeError(f"Malformed FASTQ: {path}")
            if len(seq) != len(qual):
                raise RuntimeError(f"sequence/quality length mismatch: {head.strip()}")
            n += 1
            lengths.add(len(seq))
    return n, sorted(lengths)

final_rows = read_tsv(FINAL_QC)
if len(final_rows) != 1:
    raise RuntimeError(f"Expected one mixed-sample row in {FINAL_QC}")

q = final_rows[0]
expected_pairs = int(q["expected_pairs"])
assert q["sample"] == SAMPLE
assert str(q["valid"]).lower() == "true"

n1, l1 = count_fastq(R1)
n2, l2 = count_fastq(R2)
assert n1 == n2 == expected_pairs
assert l1 == l2 == [150]

pcr1 = read_tsv(PCR1_TSV)
frags = read_tsv(FRAGMENT_TSV)
alloc = read_tsv(ALLOCATION_TSV)

pcr1_copies = sum(int(r["pcr_copies"]) for r in pcr1 if int(r["pcr_copies"]) > 0)
fragment_input = sum(int(r["fragment_input_copies"]) for r in frags)
assert pcr1_copies == fragment_input

allocated_pairs = sum(int(r["simulated_read_pairs"]) for r in alloc)
assert allocated_pairs == expected_pairs

nonzero_alloc = [r for r in alloc if int(r["simulated_read_pairs"]) > 0]

internal_summary = {
    "expected_pairs": expected_pairs,
    "R1_pairs": n1,
    "R2_pairs": n2,
    "read_length": l1[0],
    "PCR1_molecules": pcr1_copies,
    "fragment_input_molecules": fragment_input,
    "fragmentation_mass_conserved": pcr1_copies == fragment_input,
    "allocated_pairs": allocated_pairs,
    "selected_fragment_species": len(nonzero_alloc),
}
print(json.dumps(internal_summary, indent=2))


## 2. Состав смеси цепей

Четыре библиотеки объединены в один вычислительный образец. Ожидаемый бюджет по локусам определяется глубиной исходных пар ридов.


In [ ]:
EXPECTED_LOCUS_PAIRS = {
    "IGH": 1_355_378 + 1_153_931,  # IgM и IgG
    "IGK": 958_261,
    "IGL": 1_085_144,
}
assert sum(EXPECTED_LOCUS_PAIRS.values()) == expected_pairs

observed_locus_pairs = Counter()
for r in alloc:
    observed_locus_pairs[r["locus"]] += int(r["simulated_read_pairs"])

mixture_rows = []
for locus in ("IGH", "IGK", "IGL"):
    exp = EXPECTED_LOCUS_PAIRS[locus]
    obs = observed_locus_pairs[locus]
    mixture_rows.append({
        "locus": locus,
        "expected_pairs": exp,
        "observed_pairs": obs,
        "expected_pct": 100 * exp / expected_pairs,
        "observed_pct": 100 * obs / expected_pairs,
        "delta_pairs": obs - exp,
    })
    assert obs == exp

mix_path = VALIDATION_DIR / "mixture_qc.tsv"
with open(mix_path, "w", newline="") as h:
    w = csv.DictWriter(h, fieldnames=list(mixture_rows[0]), delimiter="\t")
    w.writeheader()
    w.writerows(mixture_rows)

for r in mixture_rows:
    print(r)
print("wrote", mix_path)


## 3. Детерминированная подвыборка пар

Для ускорения равномерно выбираются пары по всему FASTQ. При `VALIDATION_PAIRS=None` используются все риды.


In [ ]:
def paired_subset(r1, r2, out1, out2, total_pairs, target_pairs):
    if target_pairs is None or target_pairs >= total_pairs:
        return r1, r2, total_pairs

    stride = max(1, total_pairs // target_pairs)
    selected = 0

    tmp1 = Path(str(out1) + ".tmp")
    tmp2 = Path(str(out2) + ".tmp")

    def read_record(h):
        a = h.readline()
        if not a:
            return None
        return (a, h.readline(), h.readline(), h.readline())

    with gzip.open(r1, "rt") as h1, gzip.open(r2, "rt") as h2, \
         gzip.open(tmp1, "wt") as o1, gzip.open(tmp2, "wt") as o2:

        i = 0
        while True:
            a = read_record(h1)
            b = read_record(h2)
            if a is None or b is None:
                if a is not None or b is not None:
                    raise RuntimeError("Mate FASTQs ended at different positions")
                break

            if i % stride == 0 and selected < target_pairs:
                o1.writelines(a)
                o2.writelines(b)
                selected += 1
            i += 1

    tmp1.replace(out1)
    tmp2.replace(out2)
    return out1, out2, selected

SUB_R1 = SUBSET_DIR / f"{SAMPLE}_R1.validation.fastq.gz"
SUB_R2 = SUBSET_DIR / f"{SAMPLE}_R2.validation.fastq.gz"

if FORCE or not (SUB_R1.exists() and SUB_R2.exists()):
    VR1, VR2, validation_pair_count = paired_subset(
        R1, R2, SUB_R1, SUB_R2, expected_pairs, VALIDATION_PAIRS
    )
else:
    VR1, VR2 = SUB_R1, SUB_R2
    validation_pair_count = count_fastq(SUB_R1)[0]

print("validation pairs:", validation_pair_count)
print(VR1)
print(VR2)


## 4. Выравнивание на точные выбранные фрагменты

`04_read_allocation/*_selected_fragments.fasta` содержит последовательности, переданные в InSilicoSeq, и служит основным техническим эталоном симулятора ридов.


In [ ]:
def run(cmd, log=None):
    print("[run]", " ".join(map(str, cmd)), flush=True)
    if log is None:
        subprocess.run(list(map(str, cmd)), check=True)
    else:
        with open(log, "w") as h:
            subprocess.run(list(map(str, cmd)), stdout=h, stderr=subprocess.STDOUT, check=True)

def ensure_index(reference, prefix):
    marker = Path(str(prefix) + ".1.bt2")
    markerl = Path(str(prefix) + ".1.bt2l")
    if FORCE or not (marker.exists() or markerl.exists()):
        for p in prefix.parent.glob(prefix.name + "*.bt2*"):
            p.unlink()
        run(
            ["bowtie2-build", "--threads", str(NPROC), reference, prefix],
            prefix.parent / "bowtie2_build.log",
        )
    return prefix

def align_pair(reference, outdir, label):
    outdir.mkdir(parents=True, exist_ok=True)
    index = ensure_index(reference, outdir / "index")
    sam = outdir / f"{label}.sam"
    bam = outdir / f"{label}.bam"

    if FORCE or not bam.exists():
        run([
            "bowtie2", "--very-sensitive-local", "-p", str(NPROC),
            "-x", index, "-1", VR1, "-2", VR2, "-S", sam
        ], outdir / "bowtie2_align.log")
        run(["samtools", "sort", "-@", str(NPROC), "-o", bam, sam])
        run(["samtools", "index", bam])
        sam.unlink(missing_ok=True)
    return bam

FRAG_BAM = align_pair(FRAGMENT_FASTA, FRAG_ALIGN_DIR, "selected_fragments")
print(FRAG_BAM)


In [ ]:
def flagstat_metrics(bam):
    txt = subprocess.run(
        ["samtools", "flagstat", str(bam)],
        capture_output=True, text=True, check=True
    ).stdout
    mapped_pct = properly_paired_pct = None
    for line in txt.splitlines():
        if " mapped (" in line and "primary mapped" not in line:
            m = re.search(r"\(([\d.]+)%", line)
            if m:
                mapped_pct = float(m.group(1))
        if " properly paired (" in line:
            m = re.search(r"\(([\d.]+)%", line)
            if m:
                properly_paired_pct = float(m.group(1))
    return mapped_pct, properly_paired_pct, txt

def samtools_error_rate(bam):
    txt = subprocess.run(
        ["samtools", "stats", str(bam)],
        capture_output=True, text=True, check=True
    ).stdout
    for line in txt.splitlines():
        if "error rate:" in line:
            parts = line.split("\t")
            for i, x in enumerate(parts):
                if x.strip() == "error rate:" and i + 1 < len(parts):
                    return float(parts[i + 1])
    return None

mapped_pct, proper_pct, frag_flagstat = flagstat_metrics(FRAG_BAM)
frag_error = samtools_error_rate(FRAG_BAM)

print("fragment mapped %:", mapped_pct)
print("fragment properly paired %:", proper_pct)
print("fragment error rate:", frag_error)


## 5. Соответствие исходному фрагменту

InSilicoSeq добавляет к идентификатору референса числовые индексы рида. Из имени рида восстанавливается ожидаемый `fragment_id`. Рассчитываются доля точного совпадения идентификатора и доля выбора другого фрагмента с идентичной нуклеотидной последовательностью; второй случай не считается ошибкой симуляции.


In [ ]:
import pysam

def iter_fasta(path):
    with open(path) as h:
        name = None
        chunks = []
        for line in h:
            line = line.rstrip("\r\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name = line[1:].split()[0]
                chunks = []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

fragment_sequences = dict(iter_fasta(FRAGMENT_FASTA))

# Идентификаторы фрагментов симуляции оканчиваются на "_frag<integer>".
QNAME_RE = re.compile(r"^(.*_frag\d+)_\d+_\d+(?:/[12])?$")

def expected_fragment_id(qname):
    m = QNAME_RE.match(qname)
    return m.group(1) if m else None

def fragment_origin_metrics(bam, max_primary_reads=500_000):
    parsed = exact = equivalent = primary = 0

    with pysam.AlignmentFile(str(bam), "rb") as h:
        for r in h.fetch(until_eof=True):
            if r.is_unmapped or r.is_secondary or r.is_supplementary:
                continue
            primary += 1
            exp = expected_fragment_id(r.query_name)
            if exp is not None and exp in fragment_sequences:
                parsed += 1
                if exp == r.reference_name:
                    exact += 1
                    equivalent += 1
                elif (
                    r.reference_name in fragment_sequences
                    and fragment_sequences[exp] == fragment_sequences[r.reference_name]
                ):
                    equivalent += 1
            if primary >= max_primary_reads:
                break

    return {
        "primary_reads_checked": primary,
        "qname_origin_parse_rate": parsed / primary if primary else None,
        "exact_fragment_origin_rate": exact / parsed if parsed else None,
        "sequence_equivalent_fragment_origin_rate": equivalent / parsed if parsed else None,
    }

fragment_origin = fragment_origin_metrics(FRAG_BAM)
print(json.dumps(fragment_origin, indent=2))


## 6. Дополнительное выравнивание на исходный эталон V–J

Выравнивание на фрагменты проверяет секвенирование, а выравнивание на шаблоны — связь синтетических ридов с эталоном V–J до PCR и фрагментации. Из-за высокой гомологии иммуноглобулинов точное совпадение идентификатора шаблона строже и неоднозначнее выравнивания на фрагменты.


In [ ]:
template_metrics = None

if RUN_TEMPLATE_ALIGNMENT:
    TEMPLATE_BAM = align_pair(PRIMARY_TEMPLATES, TEMPLATE_ALIGN_DIR, "primary_templates")
    tmapped, tproper, template_flagstat = flagstat_metrics(TEMPLATE_BAM)
    terror = samtools_error_rate(TEMPLATE_BAM)
    template_metrics = {
        "mapped_pct": tmapped,
        "properly_paired_pct": tproper,
        "error_rate": terror,
    }
    print(json.dumps(template_metrics, indent=2))
else:
    print("Template alignment skipped")


## 7. Итоговая сводка проверки

- `valid=True` в `final_qc.tsv` подтверждает целостность файлов, числа ридов и длины.
- FastQC/MultiQC оценивает техническое качество PE150.
- Выравнивание на фрагменты служит основной проверкой выхода InSilicoSeq на уровне последовательностей.
- Выравнивание на шаблоны подтверждает связь с биологическим эталоном после фильтрации.
- TRUST4 запускается как отдельная стадия после успешного прохождения этих проверок.


In [ ]:
summary = {
    **internal_summary,
    "branch": BRANCH,
    "sample": SAMPLE,
    "validation_pairs": validation_pair_count,
    "locus_pairs": dict(observed_locus_pairs),
    "fragment_alignment": {
        "mapped_pct": mapped_pct,
        "properly_paired_pct": proper_pct,
        "error_rate": frag_error,
        **fragment_origin,
    },
    "template_alignment": template_metrics,
}

summary_path = VALIDATION_DIR / "validation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))
print("wrote", summary_path)
